# Adaptive Evidential Fusion (ADEF)
## Multimodal Sentiment Analysis with Evidential Deep Learning (EDL)
**Author:** Erlangga Dewa Sakti  
**Specification:** *Adaptive Evidential Multimodal Sentiment Analysis Pipeline with Gated Calibration Buffer*

---
### 1. Setup & Environment Initialization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from transformers import RobertaTokenizer, RobertaModel, CLIPVisionModel

from PIL import Image
import pandas as pd
import numpy as np
import os
import sys
import warnings
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# REPRODUCIBILITY & SEEDING
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("[INFO] Imports loaded & global seed set to 42.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


---
### 2. Configuration (`CFG` Class)
Centralized configuration management for dataset parameters, dual-stream backbone choices (RoBERTa & CLIP-ViT), ADEF hyperparameters, and Evidential Deep Learning (EDL) settings.

In [ ]:
# ============================================================
# CONFIGURATION CLASS
# ============================================================

class CFG:
    # --- Seed & System Setup ---
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 0  # 0 for safe multi-processing on Windows

    # --- Dataset & File Paths ---
    DATASET_NAME = "MVSA-Single"
    # Auto-detect data path structure (Local workspace or D:/MVSA_SINGLE)
    ROOT_DIR = "D:/MVSA_SINGLE" if os.path.exists("D:/MVSA_SINGLE") else "./data/MVSA_Single"
    DATA_DIR = os.path.join(ROOT_DIR, "data") if os.path.exists(os.path.join(ROOT_DIR, "data")) else "./data/MVSA_Single/images"
    LABEL_PATH = os.path.join(ROOT_DIR, "labelResultAllFinal.txt") if os.path.exists(os.path.join(ROOT_DIR, "labelResultAllFinal.txt")) else "./data/MVSA_Single/data_label.csv"
    OUTPUT_DIR = "./output"

    # --- Model Architecture ---
    TEXT_BACKBONE = "roberta-base"
    VISION_BACKBONE = "openai/clip-vit-base-patch32"
    HIDDEN_DIM = 768
    NUM_HEADS = 8
    NUM_CLASSES = 3
    MAX_LEN = 128
    IMAGE_SIZE = 224

    # --- CLIP Image Normalization (OpenAI Standard) ---
    NORM_MEAN = [0.48145466, 0.4578275, 0.40821073]
    NORM_STD = [0.26862954, 0.26130258, 0.27577711]

    # --- Training & Evidential Loss Hyperparameters ---
    BATCH_SIZE = 16
    EPOCHS = 5
    LR = 2e-5
    WEIGHT_DECAY = 1e-4
    MAX_ANNEAL_EPOCHS = 10
    LAMBDA_AUX = 0.2

    # --- Visualization & Logging ---
    SAVE_PLOTS = True
    PLOT_FORMAT = "png"

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
print(f"[INFO] CFG loaded. Operating on Device: {CFG.DEVICE}")
print(f"Dataset root: {CFG.ROOT_DIR}")
print(f"Output directory: {CFG.OUTPUT_DIR}")


---
### 3. Data Preparation & Multimodal Dataset
PyTorch `MultimodalDataset` handling paired image-text inputs, text tokenization with RoBERTa, CLIP image transforms, and stratified train/val/test splits.

In [ ]:
# ============================================================
# DATA PREPARATION & PYTORCH DATASET
# ============================================================

def load_and_preprocess_dataframe():
    if CFG.LABEL_PATH.endswith(".txt"):
        df = pd.read_csv(CFG.LABEL_PATH, header=0, sep=",")
        df.columns = ["id", "text_label", "image_label", "final_label"]

        # Filter conflicting text/image sentiment pairs (e.g. positive text + negative image)
        def is_valid(row):
            if row["text_label"] == "positive" and row["image_label"] == "negative":
                return False
            if row["text_label"] == "negative" and row["image_label"] == "positive":
                return False
            return True

        df = df[df.apply(is_valid, axis=1)].reset_index(drop=True)
    else:
        df = pd.read_csv(CFG.LABEL_PATH)

    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    if "label" not in df.columns and "final_label" in df.columns:
        df["label"] = df["final_label"].map(label_map)

    # Load text content per ID if reading MVSA text files
    def load_text(sample_id):
        path = os.path.join(CFG.DATA_DIR, f"{sample_id}.txt")
        if not os.path.exists(path):
            path_alt = os.path.join(CFG.ROOT_DIR, "data", f"{sample_id}.txt")
            if os.path.exists(path_alt):
                path = path_alt
            else:
                return ""
        encodings = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]
        for enc in encodings:
            try:
                with open(path, "r", encoding=enc) as f:
                    text = f.read().strip()
                    if text:
                        return text
            except Exception:
                continue
        return ""

    if "text" not in df.columns or df["text"].isnull().all():
        df["text"] = df["id"].apply(load_text)

    # Build image paths
    def get_image_path(sample_id):
        img_path = os.path.join(CFG.DATA_DIR, f"{sample_id}.jpg")
        if not os.path.exists(img_path):
            img_path_alt = os.path.join(CFG.ROOT_DIR, "data", f"{sample_id}.jpg")
            if os.path.exists(img_path_alt):
                return img_path_alt
        return img_path

    df["image_path"] = df["id"].apply(get_image_path)
    return df

df_full = load_and_preprocess_dataframe()
print(f"Total samples after filtering: {len(df_full)}")

class MultimodalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, transform, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"]) if pd.notna(row["text"]) and str(row["text"]).strip() != "" else "empty"
        image_path = row["image_path"]
        label = int(row["label"])

        # Tokenize text with RoBERTa
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # Load & transform image for CLIP-ViT
        try:
            image = Image.open(image_path).convert("RGB")
            pixel_values = self.transform(image)
        except Exception:
            pixel_values = torch.zeros(3, CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "label": torch.tensor(label, dtype=torch.long)
        }

# Preprocessing transforms & tokenizer
tokenizer = RobertaTokenizer.from_pretrained(CFG.TEXT_BACKBONE)
image_transform = transforms.Compose([
    transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG.NORM_MEAN, std=CFG.NORM_STD)
])

# Stratified 70% Train, 15% Val, 15% Test split
train_df, temp_df = train_test_split(df_full, test_size=0.30, stratify=df_full["label"], random_state=CFG.SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=CFG.SEED)

train_dataset = MultimodalDataset(train_df, tokenizer, image_transform, max_len=CFG.MAX_LEN)
val_dataset = MultimodalDataset(val_df, tokenizer, image_transform, max_len=CFG.MAX_LEN)
test_dataset = MultimodalDataset(test_df, tokenizer, image_transform, max_len=CFG.MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

print(f"[INFO] DataLoaders Ready: Train ({len(train_df)}), Val ({len(val_df)}), Test ({len(test_df)})")


---
### 4. Dual-Stream Encoders & Bidirectional Co-Attention
Modular PyTorch implementations for:
1. `TextStreamEncoder`: RoBERTa text feature extractor deriving sequence embeddings $\mathbf{H}_t \in \mathbb{R}^{B \times N \times d}$ and summary CLS embedding $\mathbf{h}_t^{cls} \in \mathbb{R}^{B \times d}$.
2. `VisionStreamEncoder`: CLIP-ViT vision feature extractor deriving patch embeddings $\mathbf{H}_v \in \mathbb{R}^{B \times M \times d}$ and pooled embedding $\mathbf{h}_v^{cls} \in \mathbb{R}^{B \times d}$.
3. `BidirectionalCoAttention`: Cross-attention mechanism computing key-query alignment matrix $\mathbf{A}^{t \to v} \in \mathbb{R}^{B \times N \times M}$, aggregating cross-attended visual features, and mean-pooling over tokens to $\mathbf{z}_{att} \in \mathbb{R}^{B \times d}$.

In [ ]:
# ============================================================
# DUAL-STREAM FEATURE EXTRACTION & CO-ATTENTION MODULES
# ============================================================

class TextStreamEncoder(nn.Module):
    """
    Text Stream Feature Extractor using HuggingFace RoBERTa.
    Extracts token sequence embeddings H_t [B, N, d] and summary CLS embedding h_t_cls [B, d].
    """
    def __init__(self, model_name=CFG.TEXT_BACKBONE, hidden_dim=CFG.HIDDEN_DIM):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        text_dim = self.roberta.config.hidden_size # 768
        self.proj = nn.Linear(text_dim, hidden_dim) if text_dim != hidden_dim else nn.Identity()

    def forward(self, input_ids, attention_mask):
        # input_ids:      [B, N]
        # attention_mask: [B, N]
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        H_t = self.proj(outputs.last_hidden_state)  # [B, N, d]
        h_t_cls = H_t[:, 0, :]                       # [B, d] (CLS token embedding)
        return H_t, h_t_cls

class VisionStreamEncoder(nn.Module):
    """
    Vision Stream Feature Extractor using HuggingFace CLIP-ViT.
    Extracts spatial patch embeddings H_v [B, M, d] and visual summary embedding h_v_cls [B, d].
    """
    def __init__(self, model_name=CFG.VISION_BACKBONE, hidden_dim=CFG.HIDDEN_DIM):
        super().__init__()
        self.clip_vit = CLIPVisionModel.from_pretrained(model_name)
        vision_dim = self.clip_vit.config.hidden_size # 768
        self.proj = nn.Linear(vision_dim, hidden_dim) if vision_dim != hidden_dim else nn.Identity()

    def forward(self, pixel_values):
        # pixel_values: [B, 3, H, W]
        outputs = self.clip_vit(pixel_values=pixel_values)
        H_v = self.proj(outputs.last_hidden_state)  # [B, M, d]
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            h_v_cls = self.proj(outputs.pooler_output) # [B, d]
        else:
            h_v_cls = H_v[:, 0, :]                    # [B, d]
        return H_v, h_v_cls

class BidirectionalCoAttention(nn.Module):
    """
    Bidirectional Multi-Head Co-Attention Module.
    Computes key-query alignment matrix A^{t->v} between text sequence H_t [B, N, d] and vision sequence H_v [B, M, d],
    aggregates cross-attended visual features, and mean-pools across token sequence dimension N to produce z_att [B, d].
    """
    def __init__(self, hidden_dim=CFG.HIDDEN_DIM, num_heads=CFG.NUM_HEADS):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert self.head_dim * num_heads == hidden_dim, "hidden_dim must be divisible by num_heads"

        self.W_Q = nn.Linear(hidden_dim, hidden_dim)
        self.W_K = nn.Linear(hidden_dim, hidden_dim)
        self.W_V = nn.Linear(hidden_dim, hidden_dim)
        self.W_O = nn.Linear(hidden_dim, hidden_dim)
        self.scale = 1.0 / np.sqrt(self.head_dim)

    def forward(self, H_t, H_v, attention_mask=None):
        # H_t: [B, N, d]
        # H_v: [B, M, d]
        B, N, _ = H_t.shape
        _, M, _ = H_v.shape

        # Linear projections & split heads: [B, num_heads, length, head_dim]
        Q = self.W_Q(H_t).view(B, N, self.num_heads, self.head_dim).transpose(1, 2) # [B, h, N, d_k]
        K = self.W_K(H_v).view(B, M, self.num_heads, self.head_dim).transpose(1, 2) # [B, h, M, d_k]
        V = self.W_V(H_v).view(B, M, self.num_heads, self.head_dim).transpose(1, 2) # [B, h, M, d_k]

        # Compute cross-attention alignment matrix A^{t->v}: [B, h, N, M]
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale                  # [B, h, N, M]
        A_t2v = F.softmax(scores, dim=-1)                                            # [B, h, N, M]

        # Aggregate values using A^{t->v}: [B, h, N, d_k]
        Z_head = torch.matmul(A_t2v, V)                                              # [B, h, N, d_k]

        # Concatenate multi-head outputs & project: [B, N, d]
        Z_cross = Z_head.transpose(1, 2).contiguous().view(B, N, self.hidden_dim)    # [B, N, d]
        Z_cross = self.W_O(Z_cross)                                                  # [B, N, d]

        # Mean-pool across tokens N to produce dense fused vector z_att: [B, d]
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand_as(Z_cross).float()  # [B, N, d]
            z_att = (Z_cross * mask_expanded).sum(dim=1) / (mask_expanded.sum(dim=1) + 1e-8) # [B, d]
        else:
            z_att = Z_cross.mean(dim=1)                                               # [B, d]

        return z_att


---
### 5. Adaptive Gated Calibration Buffer & Evidential Pipeline
Core proposed module `GatedCalibrationBuffer` combining:
- **Sub-Branch A (Informational Saliency $\Phi$):** Unimodal probes $\mathbf{W}_{probe,t}, \mathbf{W}_{probe,v}$, average normalized Shannon entropy $\bar{H} \in [0, 1]$, cross-modal cosine similarity concordance $C_{cross} \in [-1, 1]$, and non-parametric saliency factor $\Phi = (1 - \bar{H}) \cdot \left(\frac{1 + C_{cross}}{2}\right) \in [0, 1]$.
- **Sub-Branch B (Relational Context Gate $\gamma$):** Interaction context vector $\mathbf{r} = [\mathbf{z}_{att} \parallel |\mathbf{h}_t^{cls} - \mathbf{h}_v^{cls}| \parallel (\mathbf{h}_t^{cls} \odot \mathbf{h}_v^{cls})] \in \mathbb{R}^{B \times 3d}$, passing through Linear + Sigmoid to yield $\gamma = \sigma(\mathbf{W}_\gamma \mathbf{r} + b_\gamma) \in (0, 1)$.
- **Composite Calibration Factor:** $g_{cal} = \gamma \cdot \Phi \in [0, 1]$.
- **Evidential Head & Subjective Logic:** Calibrated class evidence $e_k = g_{cal} \cdot \text{Softplus}(v_{e,k})$, Dirichlet strength $S = \sum e_k + K$, belief $b_k = e_k/S$, epistemic uncertainty $u = K/S$, and expected probability point estimates $\hat{p}_k = \alpha_k/S$.

In [ ]:
# ============================================================
# GATED CALIBRATION BUFFER & COMPLETE ADEF MODEL ARCHITECTURE
# ============================================================

class GatedCalibrationBuffer(nn.Module):
    """
    Adaptive Gated Calibration Buffer (Core Proposed Module).
    Combines Non-Parametric Informational Saliency (Phi) and Parametric Relational Context Gate (gamma)
    to compute composite calibration multiplier g_cal [B, 1].
    """
    def __init__(self, hidden_dim=CFG.HIDDEN_DIM, num_classes=CFG.NUM_CLASSES, eps=1e-8):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.eps = eps

        # Sub-Branch A: Auxiliary unimodal linear probes
        self.probe_t = nn.Linear(hidden_dim, num_classes)
        self.probe_v = nn.Linear(hidden_dim, num_classes)

        # Sub-Branch B: Parametric Relational Context Gate (input dim: 3 * hidden_dim)
        self.gate_fc = nn.Linear(3 * hidden_dim, 1)

    def forward(self, z_att, h_t_cls, h_v_cls):
        # z_att:   [B, d]
        # h_t_cls: [B, d]
        # h_v_cls: [B, d]
        B, d = z_att.shape

        # --- Sub-Branch A: Non-Parametric Informational Saliency (Phi) ---
        # 1. Unimodal probe predictions
        logits_t = self.probe_t(h_t_cls)                            # [B, K]
        logits_v = self.probe_v(h_v_cls)                            # [B, K]
        p_t = F.softmax(logits_t, dim=-1)                          # [B, K]
        p_v = F.softmax(logits_v, dim=-1)                          # [B, K]

        # 2. Average Normalized Shannon Entropy: H_bar [B, 1]
        log2_k = np.log2(self.num_classes)
        entropy_t = -torch.sum(p_t * torch.log2(p_t + self.eps), dim=-1, keepdim=True) # [B, 1]
        entropy_v = -torch.sum(p_v * torch.log2(p_v + self.eps), dim=-1, keepdim=True) # [B, 1]
        H_bar = (entropy_t + entropy_v) / (2.0 * log2_k)                               # [B, 1], in [0, 1]
        H_bar = torch.clamp(H_bar, 0.0, 1.0)

        # 3. Cross-modal Cosine Similarity Concordance: C_cross [B, 1]
        cos_sim = F.cosine_similarity(h_t_cls, h_v_cls, dim=-1, eps=self.eps).unsqueeze(-1) # [B, 1], in [-1, 1]

        # 4. Non-parametric Saliency Factor: Phi [B, 1]
        saliency_phi = (1.0 - H_bar) * ((1.0 + cos_sim) / 2.0)                          # [B, 1], in [0, 1]

        # --- Sub-Branch B: Parametric Relational Context Gate (gamma) ---
        diff_feat = torch.abs(h_t_cls - h_v_cls)                    # [B, d]
        prod_feat = h_t_cls * h_v_cls                               # [B, d]
        r = torch.cat([z_att, diff_feat, prod_feat], dim=-1)         # [B, 3d]
        gate_gamma = torch.sigmoid(self.gate_fc(r))                 # [B, 1], in (0, 1)

        # --- Composite Calibration Factor: g_cal [B, 1] ---
        g_cal = gate_gamma * saliency_phi                           # [B, 1], in [0, 1]

        return {
            "g_cal": g_cal,
            "saliency_phi": saliency_phi,
            "gate_gamma": gate_gamma,
            "logits_t": logits_t,
            "logits_v": logits_v,
            "p_t": p_t,
            "p_v": p_v
        }

class ADEFGatedCalibrationModel(nn.Module):
    """
    Adaptive Evidential Multimodal Sentiment Analysis Pipeline with Gated Calibration Buffer.
    Accepts input_ids, attention_mask, pixel_values, and optional ground-truth labels.
    """
    def __init__(
        self,
        text_backbone=CFG.TEXT_BACKBONE,
        vision_backbone=CFG.VISION_BACKBONE,
        hidden_dim=CFG.HIDDEN_DIM,
        num_heads=CFG.NUM_HEADS,
        num_classes=CFG.NUM_CLASSES
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes

        # Backbones
        self.text_encoder = TextStreamEncoder(model_name=text_backbone, hidden_dim=hidden_dim)
        self.vision_encoder = VisionStreamEncoder(model_name=vision_backbone, hidden_dim=hidden_dim)

        # Cross-Attention
        self.co_attention = BidirectionalCoAttention(hidden_dim=hidden_dim, num_heads=num_heads)

        # Gated Calibration Buffer
        self.calibration_buffer = GatedCalibrationBuffer(hidden_dim=hidden_dim, num_classes=num_classes)

        # Evidential Classification Head
        self.evidential_head = nn.Linear(hidden_dim, num_classes)

        # Evidential Loss module instance
        self.loss_fn = EvidentialLoss(num_classes=num_classes, lambda_aux=CFG.LAMBDA_AUX)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None, epoch=0, max_anneal_epochs=CFG.MAX_ANNEAL_EPOCHS):
        # 1. Feature Extraction
        H_t, h_t_cls = self.text_encoder(input_ids, attention_mask)       # [B, N, d], [B, d]
        H_v, h_v_cls = self.vision_encoder(pixel_values)                   # [B, M, d], [B, d]

        # 2. Bidirectional Co-Attention Fusion
        z_att = self.co_attention(H_t, H_v, attention_mask=attention_mask) # [B, d]

        # 3. Adaptive Gated Calibration Buffer
        calib_out = self.calibration_buffer(z_att, h_t_cls, h_v_cls)       # dict
        g_cal = calib_out["g_cal"]                                          # [B, 1]

        # 4. Evidential Head & Subjective Logic Layer
        v_e = self.evidential_head(z_att)                                   # [B, K]
        evidence = g_cal * F.softplus(v_e)                                  # [B, K]
        alpha = evidence + 1.0                                              # [B, K]

        S = torch.sum(alpha, dim=-1, keepdim=True)                          # [B, 1]
        belief = evidence / S                                               # [B, K]
        uncertainty = self.num_classes / S                                  # [B, 1]
        expected_p = alpha / S                                              # [B, K]

        output = {
            "logits": v_e,
            "evidence": evidence,
            "alpha": alpha,
            "belief": belief,
            "uncertainty": uncertainty,
            "expected_p": expected_p,
            "g_cal": g_cal,
            "saliency_phi": calib_out["saliency_phi"],
            "gate_gamma": calib_out["gate_gamma"],
            "logits_t": calib_out["logits_t"],
            "logits_v": calib_out["logits_v"]
        }

        # 5. Compute loss if labels are provided
        if labels is not None:
            loss_dict = self.loss_fn(
                output=output,
                labels=labels,
                epoch=epoch,
                max_anneal_epochs=max_anneal_epochs
            )
            output["loss"] = loss_dict["loss"]
            output["loss_mse"] = loss_dict["loss_mse"]
            output["loss_kl"] = loss_dict["loss_kl"]
            output["loss_aux"] = loss_dict["loss_aux"]

        return output


---
### 6. Custom Evidential Loss Function (`EvidentialLoss`)
Dedicated loss module implementing:
$$\mathcal{L}_{EDL} = \mathcal{L}_{MSE} + \lambda_{KL} \cdot \mathcal{L}_{KL} + \lambda_{aux} \cdot (\mathcal{L}_{CE,t} + \mathcal{L}_{CE,v})$$
where $\mathcal{L}_{MSE}$ computes expected Brier/MSE error, $\mathcal{L}_{KL}$ regularizes non-target Dirichlet evidence using epoch annealing $\lambda_{KL} = \min(1.0, \text{epoch}/\text{max\_anneal\_epochs})$, and $\mathcal{L}_{aux}$ trains auxiliary unimodal probes.

In [ ]:
# ============================================================
# EVIDENTIAL LOSS MODULE
# ============================================================

class EvidentialLoss(nn.Module):
    """
    Custom Evidential Loss Function computing L_EDL = L_MSE + lambda_kl * L_KL + lambda_aux * L_aux.
    """
    def __init__(self, num_classes=CFG.NUM_CLASSES, lambda_aux=CFG.LAMBDA_AUX):
        super().__init__()
        self.num_classes = num_classes
        self.lambda_aux = lambda_aux

    def forward(self, output, labels, epoch=0, max_anneal_epochs=CFG.MAX_ANNEAL_EPOCHS):
        expected_p = output["expected_p"]        # [B, K]
        alpha = output["alpha"]                  # [B, K]
        device = alpha.device
        B = alpha.shape[0]

        # One-hot encoding of target vector
        y_onehot = F.one_hot(labels, num_classes=self.num_classes).float() # [B, K]

        # 1. Expected Mean Squared Error Loss (L_MSE)
        S = torch.sum(alpha, dim=-1, keepdim=True)                          # [B, 1]
        err = (y_onehot - expected_p) ** 2                                   # [B, K]
        var = (expected_p * (1.0 - expected_p)) / (S + 1.0)                  # [B, K]
        loss_mse = torch.mean(torch.sum(err + var, dim=-1))                 # scalar

        # 2. KL Divergence Regularization Loss (L_KL)
        alpha_tilde = y_onehot + (1.0 - y_onehot) * alpha                   # [B, K]
        S_tilde = torch.sum(alpha_tilde, dim=-1, keepdim=True)              # [B, 1]

        lgamma_k = torch.lgamma(torch.tensor(float(self.num_classes), device=device))
        first_term = torch.lgamma(S_tilde) - lgamma_k - torch.sum(torch.lgamma(alpha_tilde), dim=-1, keepdim=True)
        second_term = torch.sum((alpha_tilde - 1.0) * (torch.digamma(alpha_tilde) - torch.digamma(S_tilde)), dim=-1, keepdim=True)
        kl_div = first_term + second_term                                   # [B, 1]
        loss_kl = torch.mean(kl_div)                                        # scalar

        # KL Annealing coefficient
        lambda_kl = min(1.0, float(epoch + 1) / float(max_anneal_epochs))

        # 3. Auxiliary Cross-Entropy Loss (L_aux)
        loss_aux_t = F.cross_entropy(output["logits_t"], labels)
        loss_aux_v = F.cross_entropy(output["logits_v"], labels)
        loss_aux = 0.5 * (loss_aux_t + loss_aux_v)

        # 4. Total Combined Evidential Loss
        total_loss = loss_mse + lambda_kl * loss_kl + self.lambda_aux * loss_aux

        return {
            "loss": total_loss,
            "loss_mse": loss_mse,
            "loss_kl": loss_kl,
            "loss_aux": loss_aux,
            "lambda_kl": lambda_kl
        }


---
### 7. Training & Evaluation Pipeline
Functions for training one epoch (`train_one_epoch`), evaluating dataset splits with evidential metrics (`evaluate`), and performing complete model verification.

In [ ]:
# ============================================================
# TRAINING & EVALUATION FUNCTIONS
# ============================================================

def train_one_epoch(model, dataloader, optimizer, device, epoch, max_anneal_epochs=CFG.MAX_ANNEAL_EPOCHS):
    model.train()
    running_loss = 0.0
    running_mse = 0.0
    running_kl = 0.0
    running_aux = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{CFG.EPOCHS} [Train]")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            labels=labels,
            epoch=epoch,
            max_anneal_epochs=max_anneal_epochs
        )

        loss = output["loss"]
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        b_size = labels.size(0)
        running_loss += loss.item() * b_size
        running_mse += output["loss_mse"].item() * b_size
        running_kl += output["loss_kl"].item() * b_size
        running_aux += output["loss_aux"].item() * b_size

        pbar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "MSE": f"{output['loss_mse'].item():.4f}",
            "KL": f"{output['loss_kl'].item():.4f}"
        })

    total_samples = len(dataloader.dataset)
    return {
        "loss": running_loss / total_samples,
        "loss_mse": running_mse / total_samples,
        "loss_kl": running_kl / total_samples,
        "loss_aux": running_aux / total_samples
    }

def evaluate(model, dataloader, device, epoch=0, max_anneal_epochs=CFG.MAX_ANNEAL_EPOCHS):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_uncertainties = []
    all_g_cals = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="[Evaluating]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["label"].to(device)

            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=labels,
                epoch=epoch,
                max_anneal_epochs=max_anneal_epochs
            )

            running_loss += output["loss"].item() * labels.size(0)
            preds = torch.argmax(output["expected_p"], dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_uncertainties.extend(output["uncertainty"].squeeze(-1).cpu().numpy())
            all_g_cals.extend(output["g_cal"].squeeze(-1).cpu().numpy())

    total_samples = len(dataloader.dataset)
    avg_loss = running_loss / total_samples
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")
    mean_unc = float(np.mean(all_uncertainties))
    mean_g_cal = float(np.mean(all_g_cals))

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "mean_uncertainty": mean_unc,
        "mean_g_cal": mean_g_cal,
        "predictions": np.array(all_preds),
        "labels": np.array(all_labels),
        "uncertainties": np.array(all_uncertainties),
        "g_cals": np.array(all_g_cals)
    }


---
### 8. Verification & Pipeline Execution
Instant architecture sanity check on synthetic batch to verify tensor shapes and evidential dictionary parameters, followed by training loop execution.

In [ ]:
# ============================================================
# ARCHITECTURE VERIFICATION TEST
# ============================================================

print("[INFO] Running ADEF Gated Calibration Model verification test...")
device = CFG.DEVICE
model = ADEFGatedCalibrationModel().to(device)

# Synthetic batch for quick verification
dummy_input_ids = torch.randint(0, 1000, (2, 128)).to(device)
dummy_attention_mask = torch.ones((2, 128), dtype=torch.long).to(device)
dummy_pixel_values = torch.randn((2, 3, 224, 224)).to(device)
dummy_labels = torch.tensor([0, 2], dtype=torch.long).to(device)

out = model(
    input_ids=dummy_input_ids,
    attention_mask=dummy_attention_mask,
    pixel_values=dummy_pixel_values,
    labels=dummy_labels
)

print("\n--- Model Output Tensor Shapes ---")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"Key: {k:15s} | Shape: {list(v.shape)} | Device: {v.device}")

assert out["logits"].shape == (2, CFG.NUM_CLASSES), f"Logits shape mismatch: {out['logits'].shape}"
assert out["g_cal"].shape == (2, 1), f"g_cal shape mismatch: {out['g_cal'].shape}"
assert out["uncertainty"].shape == (2, 1), f"Uncertainty shape mismatch: {out['uncertainty'].shape}"
assert "loss" in out, "Loss missing from model output dictionary!"

print("\n[SUCCESS] Model architecture verification passed successfully!")


In [ ]:
# ============================================================
# PIPELINE TRAINING & EVALUATION RUN
# ============================================================

optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

best_val_f1 = 0.0
save_path = os.path.join(CFG.OUTPUT_DIR, "adef_gated_calibration_best.pt")

print(f"\n[INFO] Starting ADEF Training Loop for {CFG.EPOCHS} Epochs...")
for epoch in range(CFG.EPOCHS):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device, epoch, CFG.MAX_ANNEAL_EPOCHS)
    val_metrics = evaluate(model, val_loader, device, epoch, CFG.MAX_ANNEAL_EPOCHS)

    print(
        f"Epoch {epoch+1:02d}/{CFG.EPOCHS:02d} | "
        f"Train Loss: {train_metrics['loss']:.4f} (MSE: {train_metrics['loss_mse']:.4f}, KL: {train_metrics['loss_kl']:.4f}) | "
        f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val F1 (Macro): {val_metrics['f1_macro']:.4f} | Mean Unc: {val_metrics['mean_uncertainty']:.4f} | "
        f"Mean g_cal: {val_metrics['mean_g_cal']:.4f}"
    )

    if val_metrics["f1_macro"] > best_val_f1:
        best_val_f1 = val_metrics["f1_macro"]
        torch.save(model.state_dict(), save_path)
        print(f"--> Saved Best Model Checkpoint to {save_path} (Val F1: {best_val_f1:.4f})")

# Evaluate on Test Set
if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path))
    print(f"\n[INFO] Loaded best model checkpoint from {save_path}")

test_metrics = evaluate(model, test_loader, device, epoch=CFG.EPOCHS-1, max_anneal_epochs=CFG.MAX_ANNEAL_EPOCHS)
print("\n============================================================")
print("FINAL TEST SET EVALUATION RESULTS (ADEF Gated Calibration Buffer)")
print("============================================================")
print(f"Test Accuracy        : {test_metrics['accuracy']:.4f}")
print(f"Test Macro F1        : {test_metrics['f1_macro']:.4f}")
print(f"Test Weighted F1     : {test_metrics['f1_weighted']:.4f}")
print(f"Test Mean Uncertainty: {test_metrics['mean_uncertainty']:.4f}")
print(f"Test Mean g_cal      : {test_metrics['mean_g_cal']:.4f}")

print("\nClassification Report:")
print(classification_report(test_metrics["labels"], test_metrics["predictions"], target_names=["Negative", "Neutral", "Positive"]))
